# Anime Voice Training Pipeline — Colab Notebook

This notebook clones the project repository, downloads required models, uploads your data, and runs the full pipeline end-to-end.

## Before you start
1. Set the runtime to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. Replace `GITHUB_REPO_URL` below with your own GitHub repository URL.
3. Prepare your `source/` (episodes), `oped/` (optional), and `reference/` (target speaker) folders locally or in Google Drive.

In [ ]:
# ============ User Configuration ============
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/anime-voice-training-pipeline.git"
WORK_DIR = "/content/anime_voice_training"
# If using Google Drive, set your data folder path below; otherwise upload manually in the next cell.
DRIVE_DATA_DIR = "/content/drive/MyDrive/anime_voice_data"  # or None
# Aliyun DashScope API key (needed for Stage 2 ASR + SRT cleaning)
DASHSCOPE_API_KEY = ""
# ============================================

In [ ]:
import os
from pathlib import Path
import shutil

os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

# Clone the project
!git clone {GITHUB_REPO_URL} repo
PROJECT_DIR = Path(WORK_DIR) / "repo"
%cd {PROJECT_DIR}
print("Project cloned to:", PROJECT_DIR)

## 2. Install dependencies

In [ ]:
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
# UniSE requires specific dependency versions
!pip install -q transformers==4.46.3 pytorch-lightning==2.4.0

## 3. Download UniSE + BiCodec checkpoints

In [ ]:
from huggingface_hub import hf_hub_download

UNISE_DIR = PROJECT_DIR / "src" / ".."  # not used directly
EXTERNAL_UNISE = Path(WORK_DIR) / "unified-audio" / "QuarkAudio-UniSE"

# We still need the UniSE source code and checkpoints.
%cd {WORK_DIR}
!git clone https://github.com/alibaba/unified-audio.git
CKPT_DIR = EXTERNAL_UNISE / "checkpoints"
os.makedirs(CKPT_DIR / "BiCodec", exist_ok=True)
os.makedirs(CKPT_DIR / "wav2vec2-large-xlsr-53", exist_ok=True)

spark_files = [
    "config.yaml",
    "BiCodec/config.yaml",
    "BiCodec/model.safetensors",
    "wav2vec2-large-xlsr-53/README.md",
    "wav2vec2-large-xlsr-53/config.json",
    "wav2vec2-large-xlsr-53/preprocessor_config.json",
    "wav2vec2-large-xlsr-53/pytorch_model.bin",
]
for f in spark_files:
    print("Downloading", f)
    hf_hub_download(repo_id="SparkAudio/Spark-TTS-0.5B", filename=f, local_dir=str(CKPT_DIR))

print("Downloading UniSE checkpoint...")
hf_hub_download(repo_id="QuarkAudio/QuarkAudio-UniSE", filename="epoch=20-step=109367.ckpt", local_dir=str(CKPT_DIR))
print("Models ready.")

## 4. Download Mel-Band-Roformer Vocal Model

In [ ]:
from huggingface_hub import hf_hub_download

MELBAND_DIR = Path(WORK_DIR) / "Mel-Band-Roformer-Vocal-Model"
%cd {WORK_DIR}
!git clone https://github.com/KimberleyJensen/Mel-Band-Roformer-Vocal-Model.git {MELBAND_DIR}
MELBAND_CKPT = MELBAND_DIR / "MelBandRoformer.ckpt"
if not MELBAND_CKPT.exists():
    hf_hub_download(
        repo_id="KimberleyJSN/melbandroformer",
        filename="MelBandRoformer.ckpt",
        local_dir=str(MELBAND_DIR)
    )
print("Mel-Band-Roformer ready:", MELBAND_CKPT)

## 5. Prepare input data

In [ ]:
DATA_DIR = Path(WORK_DIR) / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_DATA_DIR and os.path.exists(DRIVE_DATA_DIR):
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -r {DRIVE_DATA_DIR}/* {DATA_DIR}/
    print("Copied data from Drive.")
else:
    print("Please upload your source/, oped/ (optional), and reference/ folders to:", DATA_DIR)
    print("You can use the file panel on the left to upload.")

## 6. Run the pipeline

In [ ]:
import os
os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY

TASK_DIR = DATA_DIR / "task"
SOURCE_DIR = DATA_DIR / "source"
OPED_DIR = DATA_DIR / "oped" if (DATA_DIR / "oped").exists() else None
REFERENCE_DIR = DATA_DIR / "reference"

CONFIG_PATH = PROJECT_DIR / "configs" / "default.yaml"
print("Running pipeline...")

!python main.py \
    --config {CONFIG_PATH} \
    --task-dir {TASK_DIR} \
    --source-dir {SOURCE_DIR} \
    --reference-dir {REFERENCE_DIR} \
    --stage all

## 7. Download results

In [ ]:
from google.colab import files
import zipfile

RESULT_DIR = TASK_DIR / "output" / "stage2"
ZIP_PATH = TASK_DIR / "results.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_ in os.walk(RESULT_DIR):
        for f in files_:
            fp = Path(root) / f
            zf.write(fp, arcname=fp.relative_to(TASK_DIR))

files.download(ZIP_PATH)
print("Download started.")